<!--
---
PURPOSE: Align signals to a common time grid and train models.
REQUIRES:
  - outputs/neural/session_{id}_units.parquet
  - outputs/neural/session_{id}_spike_times.npz
  - outputs/pose/session_{id}_motifs.parquet
PRODUCES:
  - outputs/fusion/session_{id}_fusion.parquet
  - outputs/models/session_{id}_model_{name}.joblib
  - outputs/models/session_{id}_metrics.json
  - outputs/models/session_{id}_interpretability.parquet
WHAT_NEXT: notebooks/09_End_to_End_Run_and_QC_Checklist.ipynb
---
-->

# 08 Neural-Behavior Fusion and Modeling

**Purpose**
Align signals to a common time grid and train models.

**Requires**
- `outputs/neural/session_{id}_units.parquet`
- `outputs/neural/session_{id}_spike_times.npz`
- `outputs/pose/session_{id}_motifs.parquet`

**Produces**
- `outputs/fusion/session_{id}_fusion.parquet`
- `outputs/models/session_{id}_model_{name}.joblib`
- `outputs/models/session_{id}_metrics.json`
- `outputs/models/session_{id}_interpretability.parquet`

**What to run next**
- `notebooks/09_End_to_End_Run_and_QC_Checklist.ipynb`

We align data to a common time grid and train predictive models.


## Environment Setup
We add the repo `src/` to the Python path so notebooks can import shared modules.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

## Prerequisite Check
We parse the notebook header and validate required artifacts before running downstream steps.

In [ ]:
from pathlib import Path
from reports import parse_notebook_header, validate_prerequisites

nb_path = Path.cwd() / "notebooks" / "08_Neural_Behavior_Fusion_and_Modeling.ipynb"
header = parse_notebook_header(nb_path)
missing = validate_prerequisites(header.get("REQUIRES", []))
if missing:
    print("Missing prerequisites:")
    for item in missing:
        print(" -", item)
else:
    print("All prerequisites satisfied.")

## Step 1: Build a common time grid
All signals are aligned to `nwb_seconds` bins.

In [ ]:
import pandas as pd
from config import get_config, make_provenance
from io_sessions import load_sessions_csv, get_session_bundle
from timebase import write_parquet_with_timebase
from modeling import build_fusion_table

cfg = get_config()
sessions = load_sessions_csv()
SESSION_ID = sessions.session_id.tolist()[:1][0]

bundle = get_session_bundle(SESSION_ID, sessions)
units, spikes = bundle.load_spikes()

fusion = None
if not spikes:
    print("No spikes available; skipping modeling for this session.")
else:
    motifs_path = cfg.outputs_dir / "pose" / f"session_{SESSION_ID}_motifs.parquet"
    motifs = pd.read_parquet(motifs_path) if motifs_path.exists() else None

    fusion = build_fusion_table(spikes, motifs, cfg.bin_size_s)
    fusion_path = cfg.outputs_dir / "fusion" / f"session_{SESSION_ID}_fusion.parquet"
    write_parquet_with_timebase(
        fusion,
        fusion_path,
        provenance=make_provenance(SESSION_ID, "nwb"),
        required_columns=["t"],
    )
    fusion.head()

## Step 2: Alignment QC
We visualize a quick sanity plot of the fused signals.

In [ ]:
from viz import plot_fusion_sanity

if fusion is not None:
    feature_cols = [c for c in fusion.columns if c != "t"]
    if feature_cols:
        plot_fusion_sanity(fusion, feature_cols[0])

## Step 3: Train and evaluate model
We fit the selected model and export metrics plus interpretability artifacts.

In [ ]:
import json
import joblib
import pandas as pd
from config import get_config, make_provenance
from modeling import fit_and_evaluate
from timebase import write_parquet_with_timebase

cfg = get_config()

if fusion is None:
    print("Fusion table missing; skipping model training.")
else:
    feature_cols = [c for c in fusion.columns if c != "t"]
    X = fusion[feature_cols]
    target_col = feature_cols[0]

    results = fit_and_evaluate(X, fusion[target_col].to_numpy(), cfg.model_name, task="count", categorical_cols=cfg.categorical_cols)
    model = results["model"]
    metrics = results["metrics"]

    model_path = cfg.outputs_dir / "models" / f"session_{SESSION_ID}_model_{cfg.model_name}.joblib"
    model_path.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump({
        "model": model,
        "model_name": cfg.model_name,
        "feature_columns": results["feature_columns"],
        "categorical_cols": cfg.categorical_cols,
        "timebase": "nwb_seconds",
    }, model_path)

    metrics_path = cfg.outputs_dir / "models" / f"session_{SESSION_ID}_metrics.json"
    metrics_payload = {
        "metrics": metrics,
        "timebase": "nwb_seconds",
        "provenance": make_provenance(SESSION_ID, "nwb"),
    }
    metrics_path.write_text(json.dumps(metrics_payload, indent=2))

    if hasattr(model, "feature_importances_"):
        imp = pd.DataFrame({"feature": results["feature_columns"], "importance": model.feature_importances_})
        imp_path = cfg.outputs_dir / "models" / f"session_{SESSION_ID}_interpretability.parquet"
        write_parquet_with_timebase(
            imp,
            imp_path,
            provenance=make_provenance(SESSION_ID, "nwb"),
            required_columns=["feature", "importance"],
        )

    model_path, metrics_path